# Setup


In [ ]:
%pip install -q -U "crewai[google-genai,tools]" ddgs

# Auth & Model


In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

from crewai import LLM

llm = LLM(
    model="gemini/gemini-3.8-flash",
    api_key=GEMINI_API_KEY,
    temperature=0.2,
    max_tokens=4096
)

print("Gemini configured successfully.")

# Search tool


In [ ]:
from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def duckduckgo_search(query: str) -> str:
    """Search the web and return titles, snippets, and URLs."""

    try:
        results = DDGS().text(query, max_results=5)

        if not results:
            return "No search results found."

        output = []

        for i, item in enumerate(results, start=1):
            output.append(
                f"Result {i}\n"
                f"Title: {item.get('title', 'N/A')}\n"
                f"Snippet: {item.get('body', 'N/A')}\n"
                f"URL: {item.get('href', 'N/A')}"
            )

        return "\n\n".join(output)

    except Exception as e:
        return f"Search error: {type(e).__name__}: {e}"

print("Search tool created.")

# Scraper tool


In [ ]:
from crewai_tools import ScrapeWebsiteTool, FileWriterTool

scraper = ScrapeWebsiteTool()
file_writer = FileWriterTool()

print("Scraper and file writer initialized.")

# Agents


In [ ]:
from crewai import Agent

site_scraper = Agent(
    role="Web Content Scraper",
    goal="Gather clear product and company details directly from the provided competitor web pages.",
    backstory="You systematically inspect company websites and only record information that is directly supported by the page content.",
    tools=[scraper],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

pricing_analyst = Agent(
    role="Pricing Specialist",
    goal="Analyze competitor pricing tiers, feature matrices, and public subscription rates.",
    backstory="You are a competitive pricing researcher who clearly documents visible prices and notes when pricing is unlisted or custom.",
    tools=[scraper],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

news_analyst = Agent(
    role="Market News Analyst",
    goal="Find verified recent press releases, product launches, and news stories for the target competitors.",
    backstory="You track corporate news with credible source links and never fabricate announcements or dates.",
    tools=[duckduckgo_search],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

strategist = Agent(
    role="Market Strategy Analyst",
    goal="Combine the research into a useful competitor comparison and SWOT analysis.",
    backstory="You are a business strategist who turns verified research into practical competitive insights.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

manager = Agent(
    role="Research Lead",
    goal="Coordinate the specialist agents, delegate research, and ensure a complete and consistent competitor report.",
    backstory="You manage competitive research projects and assign the right work to each specialist.",
    llm=llm,
    allow_delegation=True,
    verbose=True
)

print("All five agents created.")

# Task


In [ ]:
analysis_task = Task(
    description="""
    Analyze these three competitors:
    {competitors}

    Compare their products, features, publicly available pricing,
    recent announcements, strengths, weaknesses, opportunities,
    and threats.

    Include source URLs. Clearly identify unavailable information.
    Do not invent facts, prices, or announcements.
    """,
    expected_output="""
    A structured competitor analysis report with an executive summary,
    competitor comparison table, pricing and features, recent news,
    SWOT analysis, strategic recommendations, and source URLs.
    """,
    agent=manager,
    output_file="report.md"
)

# Crew


In [ ]:
from crewai import Crew, Process

competitor_crew = Crew(
    agents=[
        site_scraper,
        pricing_analyst,
        news_analyst,
        strategist
    ],
    tasks=[analysis_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True
)

print("Hierarchical Competitor Analysis Crew created successfully!")

# Input & run


In [ ]:
# Example competitor URLs: https://linear.app, https://asana.com, https://jira.com
competitors = []

for i in range(1, 4):
    url = input(f"Enter competitor {i} website URL: ").strip()

    if not url.startswith(("https://", "http://")):
        raise ValueError(f"Competitor {i} must be a valid HTTP or HTTPS URL.")

    competitors.append(url)

print("\nCompetitors to analyze:")
for url in competitors:
    print("-", url)

result = await competitor_crew.kickoff_async(
    inputs={"competitors": "\n".join(competitors)}
)

print("\nCompetitor analysis completed.")
print(result)

# Check output


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

report_path = Path("report.md")

if report_path.exists() and report_path.stat().st_size > 0:
    report = report_path.read_text(encoding="utf-8")

    print("Report created:", report_path.resolve())
    print("File size:", report_path.stat().st_size, "bytes")
    display(Markdown(report))
else:
    print("Report missing or empty.")
    print("Inspect the Crew logs for the first error.")

# Download


In [ ]:
from google.colab import files

files.download("report.md")